# Adaptive Graph Engine: Colab + Kaggle Runbook

End-to-end setup and execution on **Google Colab** or **Kaggle Notebooks**.

In [ ]:
# Detect environment and print basics
import os, sys, platform
is_colab = 'COLAB_GPU' in os.environ or 'google.colab' in sys.modules
is_kaggle = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle')
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('Colab:', is_colab, '| Kaggle:', is_kaggle)
!nvidia-smi || true


In [ ]:
# System dependencies for build
!apt-get update -qq
!apt-get install -y -qq cmake libgomp1 build-essential
!cmake --version
!nvcc --version || true


## Repository Setup
If notebook is outside your repo, set `REPO_URL` and run clone cell.
If notebook is already in repo root, keep `REPO_URL` empty.

In [ ]:
REPO_URL = ''  # e.g. https://github.com/<user>/<repo>.git
REPO_DIR = '/content/adaptive-graph-engine' if os.path.exists('/content') else '/kaggle/working/adaptive-graph-engine'
if REPO_URL.strip():
    !rm -rf "{REPO_DIR}"
    !git clone "{REPO_URL}" "{REPO_DIR}"
    WORKDIR = REPO_DIR
else:
    WORKDIR = os.getcwd()
print('WORKDIR =', WORKDIR)
%cd {WORKDIR}
!ls -la


In [ ]:
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt


In [ ]:
!bash data/download.sh
!ls -lh data


In [ ]:
!python python/preprocess.py data/facebook_combined.txt -o data/processed/facebook_combined.txt
!python python/preprocess.py data/twitter_combined.txt  -o data/processed/twitter_combined.txt
!python python/preprocess.py data/gplus_combined.txt    -o data/processed/gplus_combined.txt
!ls -lh data/processed


In [ ]:
!mkdir -p build
%cd build
!cmake ..
!cmake --build . -j4
!ls -lh


In [ ]:
DATASET = 'facebook'  # facebook | twitter | gplus
GRAPH = f'../data/processed/{DATASET}_combined.txt'
print('GRAPH =', GRAPH)


In [ ]:
!./graph_engine --graph "{GRAPH}" --algorithm all --source 0 --topk 10 --mode auto | tee ../engine_auto.log


In [ ]:
!./graph_engine --graph "{GRAPH}" --algorithm all --source 0 --topk 10 --mode auto --compare | tee ../engine_compare.log


In [ ]:
TRIALS = 3
!./benchmark_runner --graph "{GRAPH}" --dataset "{DATASET}" --trials {TRIALS} | tee ../benchmark.log
!python ../python/visualize.py --csv ../benchmarks/results/results.csv --outdir ../benchmarks/results
!ls -lh ../benchmarks/results


In [ ]:
!python ../python/analytics_report.py --log ../engine_compare.log
